In [1]:
import sys, json
from pathlib import Path

# Add the pipeline package to path (works both in Databricks and locally)
pipeline_dir = Path.cwd().parent if Path.cwd().name == "llm_pipeline" else Path.cwd()
if str(pipeline_dir) not in sys.path:
    sys.path.insert(0, str(pipeline_dir))

from llm_pipeline import CompanionPipeline, PromptEngine, init_client, validate_intent
from llm_pipeline.config import ACTION_SET, ACTION_NAMES, COMPANION_PROFILES

# Load test scenarios
fixtures_path = Path("fixtures/test_scenarios.json")
if not fixtures_path.exists():
    # Try relative to notebook location in Databricks
    fixtures_path = Path("/Workspace/Users/shawn.xu@tempered.ai/pz_AI_NPC/runtime/llm_pipeline/fixtures/test_scenarios.json")

with open(fixtures_path, "r") as f:
    scenarios = json.load(f)["scenarios"]

print(f"Loaded {len(scenarios)} test scenarios:")
for s in scenarios:
    print(f"  - {s['name']}: {s['description']} (expected: {s['expected_action']})")


Loaded 12 test scenarios:
  - player_follow: Basic follow command (expected: continue_follow)
  - player_pickup_axe: Player asks to pick up specific item (expected: pick_up_item)
  - event_noise: Strange noise heard, should investigate (expected: investigate)
  - event_threat_distant: Zombies spotted far away, should retreat to player (expected: retreat)
  - event_single_zombie_close: Single zombie very close, high health - should defend (expected: defend)
  - event_horde_low_health: Multiple zombies, companion is low health - must retreat (expected: retreat)
  - player_wait: Player tells companion to wait (expected: wait)
  - player_loot_building: Player asks to search a building (expected: loot)
  - player_ambiguous: Ambiguous player text - should default to continue_follow (expected: continue_follow)
  - periodic_safe: Periodic tick with no threats - should continue following (expected: continue_follow)
  - player_barricade: Player asks to fortify position (expected: barricade)
  - 

In [2]:
SCENARIO_IDX = 4  # Change this to preview different scenarios
PROFILE = "cautious_survivor"  # Try: neutral, cautious_survivor, aggressive_fighter, resourceful_scavenger

engine = PromptEngine(profile_key=PROFILE)
scenario = scenarios[SCENARIO_IDX]

print(f"=== Scenario: {scenario['name']} ===")
print(f"=== Profile: {PROFILE} ({COMPANION_PROFILES[PROFILE]['name']}) ===")
print()
print("--- SYSTEM PROMPT ---")
print(engine.build_system_prompt())
print()
print("--- USER PROMPT ---")
print(engine.build_user_prompt(scenario["request"]))
print()
print(f"Expected action: {scenario['expected_action']}")

=== Scenario: event_single_zombie_close ===
=== Profile: cautious_survivor (Elena) ===

--- SYSTEM PROMPT ---
You are the decision-making module for a human survivor companion NPC in Project Zomboid.

Your job: given the current situation, choose **exactly one** action from the
fixed set below that best fits. You are answering a structured decision question,
not roleplaying or writing dialogue.


Companion identity:
- Name: Elena
- Personality: Cautious and methodical. Prefers to avoid conflict when possible. Prioritizes safety and resource conservation.
Let this personality subtly influence your decisions when multiple actions seem equally valid.

Available actions:
- continue_follow: Nothing changes, keep following the player
- move_to: Move to the given coordinates (target_x, target_y)
- investigate: Go check out something that caught attention (a noise, a suspicious spot); needs target_x/target_y
- pick_up_item: Pick up an item; needs target_id identifying the item
- retreat: Fall 

In [3]:

test_outputs = [
    # Valid outputs
    {"action": "continue_follow", "target_x": None, "target_y": None, "target_id": None, "reason": "player said follow"},
    {"action": "investigate", "target_x": 120, "target_y": 340, "target_id": None, "reason": "noise at location"},
    {"action": "pick_up_item", "target_x": None, "target_y": None, "target_id": "axe_01", "reason": "player asked for axe"},
    # Edge cases: should trigger warnings
    {"action": "wait", "target_x": 100, "target_y": 200, "target_id": None, "reason": "waiting"},  # unnecessary coords
    {"action": "investigate", "target_x": None, "target_y": None, "target_id": None, "reason": "noise"},  # missing required coords
    {"action": "pick_up_item", "target_x": None, "target_y": None, "target_id": None, "reason": "grab it"},  # missing target_id
    # Invalid outputs: should fallback safely
    {"action": "fly_away", "target_x": None, "target_y": None, "target_id": None, "reason": "escape"},  # invalid action
    None,  # parse failure
]

print(f"{'Output':<50} {'Valid':<6} {'Action':<18} {'Intent':<12} {'Conf':<5} {'Warnings'}")
print("-" * 120)
for output in test_outputs:
    result = validate_intent(output, raw_content=str(output))
    label = str(output)[:48] if output else "None (parse failure)"
    warnings = "; ".join(result.warnings) if result.warnings else "-"
    print(f"{label:<50} {str(result.valid):<6} {result.action:<18} {result.intent:<12} {result.confidence:<5} {warnings}")


Validation failed: unknown action 'fly_away'
Validation failed: non-JSON output


Output                                             Valid  Action             Intent       Conf  Warnings
------------------------------------------------------------------------------------------------------------------------
{'action': 'continue_follow', 'target_x': None,    True   continue_follow    FOLLOW       1.0   -
{'action': 'investigate', 'target_x': 120, 'targ   True   investigate        NONE         1.0   -
{'action': 'pick_up_item', 'target_x': None, 'ta   True   pick_up_item       COLLECT_RESOURCE 1.0   -
{'action': 'wait', 'target_x': 100, 'target_y':    True   wait               WAIT         1.0   wait should not have coordinates; ignoring (100, 200)
{'action': 'investigate', 'target_x': None, 'tar   True   investigate        NONE         0.5   investigate requires target_x/target_y but got (None, None)
{'action': 'pick_up_item', 'target_x': None, 'ta   True   pick_up_item       COLLECT_RESOURCE 0.5   pick_up_item requires target_id but none provided
{'action': 'fly_away

In [2]:
pipe = CompanionPipeline(profile="neutral")

In [8]:
# --- Run single scenario ---
SCENARIO_IDX = 0
scenario = scenarios[SCENARIO_IDX]

print(f"Running: {scenario['name']} (expected: {scenario['expected_action']})")
print()

result = pipe.decide(scenario["request"])
summary = result.summary()

print(f"Action:     {summary['action']}")
print(f"Intent:     {summary['intent']}")
print(f"Valid:      {summary['valid']}")
print(f"Confidence: {summary['confidence']}")
print(f"Reason:     {summary['reason']}")
print(f"Elapsed:    {summary['elapsed_ms']}ms")
print(f"Tokens:     {summary['tokens']}")
if summary['warnings']:
    print(f"Warnings:   {summary['warnings']}")
if summary['error']:
    print(f"ERROR:      {summary['error']}")
print()
match = '✓ PASS' if summary['action'] == scenario['expected_action'] else '✗ FAIL'
print(f"Match: {match}")

Running: player_follow (expected: continue_follow)

Action:     continue_follow
Intent:     FOLLOW
Valid:      True
Confidence: 1.0
Reason:     player explicitly asked to keep following
Elapsed:    854ms
Tokens:     665+31

Match: ✓ PASS


In [10]:
pipe = CompanionPipeline(profile="neutral")

requests = [s["request"] for s in scenarios]
labels = [s["name"] for s in scenarios]
expected = [s["expected_action"] for s in scenarios]

results = pipe.batch_decide(requests, labels=labels)

# Results table
print(f"{'Scenario':<30} {'Expected':<20} {'Got':<20} {'Match':<6} {'Ms':<6} {'Warnings'}")
print("=" * 110)

passed = 0
for r, exp in zip(results, expected):
    match = "✓" if r["action"] == exp else "✗"
    if r["action"] == exp:
        passed += 1
    warnings = "; ".join(r["warnings"]) if r["warnings"] else "-"
    print(f"{r['label']:<30} {exp:<20} {r['action']:<20} {match:<6} {r['elapsed_ms']:<6} {warnings}")

print()
print(f"Pass rate: {passed}/{len(scenarios)} ({100*passed/len(scenarios):.0f}%)")
print(f"Total time: {sum(r['elapsed_ms'] for r in results)}ms")
print(f"Avg latency: {sum(r['elapsed_ms'] for r in results) / len(results):.0f}ms")


Scenario                       Expected             Got                  Match  Ms     Warnings
player_follow                  continue_follow      continue_follow      ✓      1063   -
player_pickup_axe              pick_up_item         pick_up_item         ✓      1707   -
event_noise                    investigate          investigate          ✓      1791   investigate requires target_x/target_y but got (None, None)
event_threat_distant           retreat              investigate          ✗      2646   investigate requires target_x/target_y but got (None, None)
event_single_zombie_close      defend               investigate          ✗      1727   investigate requires target_x/target_y but got (None, None)
event_horde_low_health         retreat              retreat              ✓      2985   -
player_wait                    wait                 wait                 ✓      1192   -
player_loot_building           loot                 loot                 ✓      1728   loot should not have

In [3]:

THREAT_SCENARIO = scenarios[4]  # event_single_zombie_close
print(f"Scenario: {THREAT_SCENARIO['name']} - {THREAT_SCENARIO['description']}")
print()

for profile_key, profile in COMPANION_PROFILES.items():
    pipe = CompanionPipeline(profile=profile_key)
    result = pipe.decide(THREAT_SCENARIO["request"])
    s = result.summary()
    print(f"  [{profile_key:>25}] {profile['name']:<12} -> {s['action']:<18} reason: {s['reason']}")


Scenario: event_single_zombie_close - Single zombie very close, high health - should defend

  [        cautious_survivor] Elena        -> investigate        reason: zombie_spotted event detected at (103, 203), which is the closest threat to the player's current position; prioritize safety and avoid unnecessary retreats when a single threat is present.
  [       aggressive_fighter] Marcus       -> investigate        reason: Player is at the same location as the zombie, but the event report indicates a threat is approaching; investigating the area is the best course of action before engaging in combat.
  [    resourceful_scavenger] Kit          -> investigate        reason: zombie_spotted event detected at (103, 203), which is close enough to investigate for supplies
  [                  neutral] Companion    -> investigate        reason: Player is at 100,200 and the only threat is a zombie at 103,203 (3 tiles away), which is too far to engage immediately; investigating the area is the 